In [1]:


import pandas as pd
import numpy as np
import gc
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# 1. ЗАГРУЗКА
# =============================================================================

BASE = "/kaggle/input/datasets/zhansdev/ninjas/"

train_users     = pd.read_csv(BASE + "train_users.csv")
train_props     = pd.read_csv(BASE + "train_users_properties.csv")
train_quizzes   = pd.read_csv(BASE + "train_users_quizzes.csv",                 low_memory=False)
train_purchases = pd.read_csv(BASE + "train_users_purchases.csv")
train_txn       = pd.read_csv(BASE + "train_users_transaction_attempts_v1.csv", low_memory=False)

test_users      = pd.read_csv(BASE + "test_users.csv")
test_props      = pd.read_csv(BASE + "test_users_properties.csv")
test_quizzes    = pd.read_csv(BASE + "test_users_quizzes.csv",                  low_memory=False)
test_purchases  = pd.read_csv(BASE + "test_users_purchases.csv")
test_txn        = pd.read_csv(BASE + "test_users_transaction_attempts_v1.csv",  low_memory=False)

print(f"Train: {train_users.shape[0]} users | Test: {test_users.shape[0]} users")
print("Target:\n", train_users['churn_status'].value_counts())

# =============================================================================
# 2. GENERATIONS — агрегируем чанками, не грузим весь файл в память
# =============================================================================

def load_and_agg_generations(path):

    CHUNK = 300_000
    parts = []

    for chunk in pd.read_csv(path, low_memory=False, chunksize=CHUNK):
        chunk.drop(columns=['Unnamed: 0'], errors='ignore', inplace=True)

        # Даты
        chunk['created_at']   = pd.to_datetime(chunk['created_at'],   utc=True, errors='coerce')
        chunk['completed_at'] = pd.to_datetime(chunk['completed_at'], utc=True, errors='coerce')
        chunk['failed_at']    = pd.to_datetime(chunk['failed_at'],    utc=True, errors='coerce')
        chunk['credit_cost']  = pd.to_numeric(chunk['credit_cost'],   errors='coerce').fillna(0)

        # Флаги
        chunk['is_completed']    = (chunk['status'] == 'completed').astype(int)
        chunk['is_failed']       = chunk['failed_at'].notna().astype(int)
        chunk['is_nsfw']         = (chunk['status'] == 'nsfw').astype(int)
        chunk['gen_duration_sec'] = (
            chunk['completed_at'] - chunk['created_at']
        ).dt.total_seconds().clip(lower=0)
        chunk['created_hour']    = chunk['created_at'].dt.hour
        chunk['is_night']        = (chunk['created_hour'] < 6).astype(int)

        # Частичная агрегация по user_id
        part = chunk.groupby('user_id').agg(
            gen_count              = ('generation_id',     'count'),
            gen_credit_total       = ('credit_cost',       'sum'),
            gen_credit_sum_sq      = ('credit_cost',       lambda x: (x**2).sum()),  # для std
            gen_credit_max         = ('credit_cost',       'max'),
            gen_completed_count    = ('is_completed',      'sum'),
            gen_failed_count       = ('is_failed',         'sum'),
            gen_nsfw_count         = ('is_nsfw',           'sum'),
            gen_duration_sum       = ('gen_duration_sec',  'sum'),
            gen_duration_max       = ('gen_duration_sec',  'max'),
            gen_unique_types       = ('generation_type',   'nunique'),
            gen_unique_resolutions = ('resolution',        'nunique'),
            gen_night_count        = ('is_night',          'sum'),
        ).reset_index()

        # Топ generation_type и resolution для этого чанка
        top_type = (chunk.groupby('user_id')['generation_type']
                    .agg(lambda x: x.value_counts().idxmax())
                    .reset_index().rename(columns={'generation_type': 'gen_top_type_chunk'}))
        top_res  = (chunk[chunk['resolution'].notna()]
                    .groupby('user_id')['resolution']
                    .agg(lambda x: x.value_counts().idxmax())
                    .reset_index().rename(columns={'resolution': 'gen_top_resolution_chunk'}))

        part = part.merge(top_type, on='user_id', how='left')
        part = part.merge(top_res,  on='user_id', how='left')
        part['gen_top_type_chunk']       = part['gen_top_type_chunk'].fillna('unknown')
        part['gen_top_resolution_chunk'] = part['gen_top_resolution_chunk'].fillna('unknown')

        parts.append(part)
        del chunk
        gc.collect()

    # Объединяем все частичные агрегаты
    all_parts = pd.concat(parts, ignore_index=True)
    del parts; gc.collect()

    # Финальная агрегация по user_id
    agg = all_parts.groupby('user_id').agg(
        gen_count              = ('gen_count',              'sum'),
        gen_credit_total       = ('gen_credit_total',       'sum'),
        gen_credit_max         = ('gen_credit_max',         'max'),
        gen_completed_count    = ('gen_completed_count',    'sum'),
        gen_failed_count       = ('gen_failed_count',       'sum'),
        gen_nsfw_count         = ('gen_nsfw_count',         'sum'),
        gen_duration_sum       = ('gen_duration_sum',       'sum'),
        gen_duration_max       = ('gen_duration_max',       'max'),
        gen_unique_types       = ('gen_unique_types',       'max'),
        gen_unique_resolutions = ('gen_unique_resolutions', 'max'),
        gen_night_count        = ('gen_night_count',        'sum'),
        # Топ type/resolution — берём моду среди частей (упрощение: берём первую)
        gen_top_type           = ('gen_top_type_chunk',     lambda x: x.value_counts().idxmax()),
        gen_top_resolution     = ('gen_top_resolution_chunk', lambda x: x.value_counts().idxmax()),
    ).reset_index()

    # Производные метрики
    agg['gen_credit_mean']     = agg['gen_credit_total'] / agg['gen_count'].replace(0, np.nan)
    agg['gen_duration_mean']   = agg['gen_duration_sum']  / agg['gen_count'].replace(0, np.nan)
    agg['gen_completion_rate'] = agg['gen_completed_count'] / agg['gen_count'].replace(0, np.nan)
    agg['gen_failure_rate']    = agg['gen_failed_count']    / agg['gen_count'].replace(0, np.nan)
    agg['gen_nsfw_rate']       = agg['gen_nsfw_count']      / agg['gen_count'].replace(0, np.nan)
    for col in ['gen_credit_mean', 'gen_duration_mean', 'gen_completion_rate',
                'gen_failure_rate', 'gen_nsfw_rate']:
        agg[col] = agg[col].fillna(0)

    agg.drop(columns=['gen_duration_sum'], inplace=True)
    del all_parts; gc.collect()

    return agg


train_gens_agg = load_and_agg_generations(BASE + "train_users_generations.csv/train_users_generations.csv")
print(f"✓ train_gens_agg: {train_gens_agg.shape}")

test_gens_agg = load_and_agg_generations(BASE + "test_users_generations.csv")
print(f"✓ test_gens_agg: {test_gens_agg.shape}")

# =============================================================================
# 3. ЧИСТКА остальных таблиц
# =============================================================================

for df in [train_users, test_users, train_props, test_props,
           train_quizzes, test_quizzes, train_purchases, test_purchases,
           train_txn, test_txn]:
    df.drop(columns=['Unnamed: 0'], errors='ignore', inplace=True)

# Properties
for df in [train_props, test_props]:
    df['subscription_start_date'] = pd.to_datetime(
        df['subscription_start_date'], utc=True, errors='coerce')
    df['country_code'] = df['country_code'].fillna('UNKNOWN')

# Quizzes
for df in [train_quizzes, test_quizzes]:
    df.drop(columns=['flow_type'], errors='ignore', inplace=True)
    df['_filled'] = df.notna().sum(axis=1)
    df.sort_values('_filled', inplace=True)
    df.drop_duplicates('user_id', keep='last', inplace=True)
    df.drop(columns=['_filled'], inplace=True)
    for col in ['source','team_size','experience','usage_plan','frustration','first_feature','role']:
        if col in df.columns:
            df[col] = df[col].fillna('unknown')

# Purchases
for df in [train_purchases, test_purchases]:
    df['purchase_time'] = pd.to_datetime(df['purchase_time'], utc=True, errors='coerce')

# Transactions
for df in [train_txn, test_txn]:
    df['transaction_time'] = pd.to_datetime(df['transaction_time'], utc=True, errors='coerce')
    df['amount_in_usd'] = df['amount_in_usd'].clip(upper=df['amount_in_usd'].quantile(0.999))
    df['billing_address_country'] = df['billing_address_country'].fillna('UNKNOWN')
    df['card_country']             = df['card_country'].fillna('UNKNOWN')
    df['bank_name']                = df['bank_name'].fillna('NO_BANK')
    df['bank_country']             = df['bank_country'].fillna('UNKNOWN')
    for col in ['is_prepaid','is_virtual','is_business']:
        df[col] = df[col].map({True:1,False:0,'True':1,'False':0}).fillna(0).astype(int)

# =============================================================================
# 4. FEATURE ENGINEERING
# =============================================================================

def build_features(users_df, props_df, quizzes_df, purchases_df, txn_df, gens_agg_df):
    df = users_df[['user_id']].copy()

    # Properties
    props = props_df.copy()
    props['subscription_date_rank'] = props['subscription_start_date'].rank(method='dense')
    props['sub_start_month']        = props['subscription_start_date'].dt.month
    props['sub_start_dow']          = props['subscription_start_date'].dt.dayofweek
    df = df.merge(props[['user_id','subscription_plan','country_code',
                          'subscription_date_rank','sub_start_month','sub_start_dow']],
                  on='user_id', how='left')

    # Quizzes
    quiz_cols = [c for c in ['user_id','source','team_size','experience',
                              'usage_plan','frustration','first_feature','role']
                 if c in quizzes_df.columns]
    df = df.merge(quizzes_df[quiz_cols], on='user_id', how='left')
    for col in quiz_cols[1:]:
        if col in df.columns:
            df[col] = df[col].fillna('unknown')

    # Purchases
    pagg = purchases_df.groupby('user_id').agg(
        purchase_count        = ('transaction_id',          'count'),
        purchase_total        = ('purchase_amount_dollars',  'sum'),
        purchase_avg          = ('purchase_amount_dollars',  'mean'),
        purchase_max          = ('purchase_amount_dollars',  'max'),
        purchase_types_unique = ('purchase_type',            'nunique'),
        has_credits_package   = ('purchase_type', lambda x: int('Credits package'     in x.values)),
        has_subscription      = ('purchase_type', lambda x: int('Subscription Create' in x.values)),
        has_upsell            = ('purchase_type', lambda x: int('Upsell'              in x.values)),
        has_reactivation      = ('purchase_type', lambda x: int('Reactivation'        in x.values)),
    ).reset_index()
    df = df.merge(pagg, on='user_id', how='left')
    df[[c for c in pagg.columns if c != 'user_id']] = \
        df[[c for c in pagg.columns if c != 'user_id']].fillna(0)

    # Transactions
    t = txn_df.copy()
    t['is_failed'] = t['failure_code'].notna().astype(int)
    tagg = t.groupby('user_id').agg(
        txn_count            = ('transaction_id',          'count'),
        txn_failed           = ('is_failed',               'sum'),
        txn_success          = ('is_failed',  lambda x:   (x==0).sum()),
        txn_avg_amount       = ('amount_in_usd',           'mean'),
        txn_max_amount       = ('amount_in_usd',           'max'),
        txn_total_amount     = ('amount_in_usd',           'sum'),
        txn_unique_cards     = ('card_brand',              'nunique'),
        txn_unique_countries = ('billing_address_country', 'nunique'),
        txn_3ds_count        = ('is_3d_secure',            'sum'),
        txn_prepaid          = ('is_prepaid',              'sum'),
        txn_card_declined    = ('failure_code', lambda x: (x=='card_declined').sum()),
        txn_expired_card     = ('failure_code', lambda x: (x=='expired_card').sum()),
        txn_incorrect_cvc    = ('failure_code', lambda x: (x=='incorrect_cvc').sum()),
        txn_invalid_cvc      = ('failure_code', lambda x: (x=='invalid_cvc').sum()),
        txn_processing_error = ('failure_code', lambda x: (x=='processing_error').sum()),
        txn_auth_required    = ('failure_code', lambda x: (x=='authentication_required').sum()),
        txn_incorrect_number = ('failure_code', lambda x: (x=='incorrect_number').sum()),
    ).reset_index()
    tagg['txn_failure_rate'] = (tagg['txn_failed'] / tagg['txn_count'].replace(0,np.nan)).fillna(0)
    top_fail = (t[t['failure_code'].notna()]
                .groupby('user_id')['failure_code']
                .agg(lambda x: x.value_counts().idxmax())
                .reset_index().rename(columns={'failure_code':'top_failure_code'}))
    tagg = tagg.merge(top_fail, on='user_id', how='left')
    tagg['top_failure_code'] = tagg['top_failure_code'].fillna('none')
    df = df.merge(tagg, on='user_id', how='left')
    df[[c for c in tagg.columns if c != 'user_id']] = \
        df[[c for c in tagg.columns if c != 'user_id']].fillna(0)
    df['top_failure_code'] = df['top_failure_code'].replace(0, 'none')

    # Generations
    df = df.merge(gens_agg_df, on='user_id', how='left')
    gen_num_cols = [c for c in gens_agg_df.columns
                    if c != 'user_id' and gens_agg_df[c].dtype in [np.float64, np.int64]]
    df[gen_num_cols] = df[gen_num_cols].fillna(0)
    for col in ['gen_top_type','gen_top_resolution']:
        if col in df.columns:
            df[col] = df[col].fillna('unknown')

    # Cross-фичи
    df['gen_per_dollar']    = (df['gen_count'] / df['purchase_total'].replace(0,np.nan)).fillna(0)
    df['credit_per_dollar'] = (df['gen_credit_total'] / df['purchase_total'].replace(0,np.nan)).fillna(0)
    df['gen_per_txn_fail']  = (df['gen_count'] / df['txn_failed'].replace(0,np.nan)).fillna(0)
    for col in ['gen_per_dollar','credit_per_dollar']:
        q99 = df[col].quantile(0.99)
        df[col] = df[col].clip(upper=q99)

    return df


train_feats = build_features(
    train_users, train_props, train_quizzes, train_purchases, train_txn, train_gens_agg
)
train_feats['churn_status'] = (
    train_users.set_index('user_id').loc[train_feats['user_id']]['churn_status'].values
)

test_feats = build_features(
    test_users, test_props, test_quizzes, test_purchases, test_txn, test_gens_agg
)

print(f"✓ Train: {train_feats.shape} | Test: {test_feats.shape}")

# =============================================================================
# 5. КОДИРОВАНИЕ
# =============================================================================

LABEL_MAP     = {'not_churned': 0, 'invol_churn': 1, 'vol_churn': 2}
LABEL_MAP_INV = {v: k for k, v in LABEL_MAP.items()}

cat_cols = [c for c in [
    'subscription_plan','country_code','source','team_size','experience',
    'usage_plan','frustration','first_feature','role',
    'top_failure_code','gen_top_type','gen_top_resolution'
] if c in train_feats.columns]

for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train_feats[col], test_feats[col]], axis=0).astype(str)
    le.fit(combined)
    train_feats[col] = le.transform(train_feats[col].astype(str))
    test_feats[col]  = le.transform(test_feats[col].astype(str))

FEATURE_COLS = [c for c in train_feats.columns if c not in ['user_id','churn_status']]
X      = train_feats[FEATURE_COLS]
y      = train_feats['churn_status'].map(LABEL_MAP)
X_test = test_feats.reindex(columns=FEATURE_COLS, fill_value=0)

print(f"✓ Фичей итого: {len(FEATURE_COLS)}")

# =============================================================================
# 6. ОБУЧЕНИЕ — LightGBM 5-Fold CV
# =============================================================================

N_SPLITS   = 5
oof_preds  = np.zeros((len(X), 3))
test_preds = np.zeros((len(X_test), 3))
models     = []
fold_f1s   = []

lgb_params = {
    'objective':         'multiclass',
    'num_class':         3,
    'metric':            'multi_logloss',
    'n_estimators':      2000,
    'learning_rate':     0.02,
    'num_leaves':        127,
    'min_child_samples': 20,
    'subsample':         0.8,
    'subsample_freq':    1,
    'colsample_bytree':  0.8,
    'reg_alpha':         0.1,
    'reg_lambda':        0.1,
    'class_weight':      'balanced',
    'random_state':      42,
    'n_jobs':            -1,
    'verbose':           -1,
}

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
print(f"\n── LightGBM {N_SPLITS}-Fold CV ──")

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(150, verbose=False),
            lgb.log_evaluation(9999),
        ]
    )

    val_proba = model.predict_proba(X_val)
    oof_preds[val_idx] = val_proba
    test_preds += model.predict_proba(X_test) / N_SPLITS
    models.append(model)

    f1_wt  = f1_score(y_val, np.argmax(val_proba, axis=1), average='weighted')
    f1_mac = f1_score(y_val, np.argmax(val_proba, axis=1), average='macro')
    fold_f1s.append(f1_wt)
    print(f"  Fold {fold+1}/{N_SPLITS} | F1 weighted: {f1_wt:.4f} | F1 macro: {f1_mac:.4f}"
          f" | best iter: {model.best_iteration_}")

# =============================================================================
# 7. МЕТРИКИ OOF
# =============================================================================

oof_labels = np.argmax(oof_preds, axis=1)
y_true     = y.values

f1_mac = f1_score(y_true, oof_labels, average='macro')
f1_wt  = f1_score(y_true, oof_labels, average='weighted')
f1_per = f1_score(y_true, oof_labels, average=None)

print("\n" + "="*65)
print("  OOF МЕТРИКИ")
print("="*65)
print(f"\n📊 F1 weighted : {f1_wt:.4f}   ← метрика хакатона")
print(f"   F1 macro    : {f1_mac:.4f}")
print(f"   CV mean±std : {np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}")
print(f"\n📋 F1 по классам:")
print(f"   not_churned : {f1_per[0]:.4f}  — лояльные")
print(f"   invol_churn : {f1_per[1]:.4f}  — платёжные сбои")
print(f"   vol_churn   : {f1_per[2]:.4f}  — осознанный отказ")
print(f"\n{classification_report(y_true, oof_labels, target_names=['not_churned','invol_churn','vol_churn'])}")

cm = confusion_matrix(y_true, oof_labels)
print("🔢 Confusion Matrix:")
print(pd.DataFrame(cm,
    index  =['True: not_churned','True: invol_churn','True: vol_churn'],
    columns=['Pred: not_churned','Pred: invol_churn','Pred: vol_churn']
))

feat_imp = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': models[-1].feature_importances_
}).sort_values('importance', ascending=False)
print(f"\n📌 Топ-25 важных признаков:")
print(feat_imp.head(25).to_string(index=False))

# =============================================================================
# 8. САБМИТ
# =============================================================================

test_label_names = [LABEL_MAP_INV[l] for l in np.argmax(test_preds, axis=1)]

submission = pd.DataFrame({
    'user_id':      test_feats['user_id'],
    'churn_status': test_label_names,
})
submission.to_csv('submission.csv', index=False)

submission_proba = pd.DataFrame({
    'user_id':           test_feats['user_id'],
    'churn_status':      test_label_names,
    'prob_not_churned':  np.round(test_preds[:, 0], 4),
    'prob_invol_churn':  np.round(test_preds[:, 1], 4),
    'prob_vol_churn':    np.round(test_preds[:, 2], 4),
})
submission_proba.to_csv('submission_with_proba.csv', index=False)

print(f"\n submission.csv готов — {len(submission)} строк")
print(submission['churn_status'].value_counts())


Загружаем данные...
Train: 90000 users | Test: 7000 users
Target:
 churn_status
not_churned    45000
invol_churn    22500
vol_churn      22500
Name: count, dtype: int64

Загружаем и агрегируем train_generations (5 ГБ, подождите ~2-3 мин)...
✓ train_gens_agg: (85616, 18)
Загружаем и агрегируем test_generations...
✓ test_gens_agg: (6687, 18)

Чистим данные...
✓ Чистка завершена

Строим фичи train...
Строим фичи test...
✓ Train: (90000, 62) | Test: (7000, 61)
✓ Фичей итого: 60

── LightGBM 5-Fold CV ──
  Fold 1/5 | F1 weighted: 0.5485 | F1 macro: 0.5219 | best iter: 1180
  Fold 2/5 | F1 weighted: 0.5498 | F1 macro: 0.5214 | best iter: 1431
  Fold 3/5 | F1 weighted: 0.5467 | F1 macro: 0.5189 | best iter: 1347
  Fold 4/5 | F1 weighted: 0.5448 | F1 macro: 0.5187 | best iter: 1113
  Fold 5/5 | F1 weighted: 0.5496 | F1 macro: 0.5211 | best iter: 1400

  OOF МЕТРИКИ

📊 F1 weighted : 0.5479   ← метрика хакатона
   F1 macro    : 0.5204
   CV mean±std : 0.5479 ± 0.0019

📋 F1 по классам:
   not_chu